In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import sys
import tomllib
import os
from datetime import datetime
from pathlib import Path
from sqlalchemy import create_engine, text
import polars as pl


In [26]:
today = datetime.now().strftime("%Y-%m-%d")
CURRENT_NOTEBOOK_DIR = Path.cwd()  # Currnet Dir
ROOT_DIR = CURRENT_NOTEBOOK_DIR.parent # Root Dir

if str(ROOT_DIR) not in sys.path:    # 내 프로젝트 최상위 폴더"seoul_public_bike" 위치를 강제로 등록
    sys.path.append(str(ROOT_DIR))




from utils.openapi_client import OpenApiClient

In [27]:
toml_path = ROOT_DIR / "config" / "dev.toml"
base_fact_path = ROOT_DIR / "datalake" / "seoul-public-bike" / "fact"

with open(toml_path, "rb") as f:
    config = tomllib.load(f)



service_name = config["seoul_api_fact"]["service_name"]  # "bikeList"
api_key = config["seoul_api_global"]["api_key"]

param_start_date = "20250101"
param_end_date   = "20250103"
start_date_formatted = datetime.strptime(param_start_date, "%Y%m%d").strftime("%Y-%m-%d")
end_date_formatted   = datetime.strptime(param_end_date, "%Y%m%d").strftime("%Y-%m-%d")

open_api_client:OpenApiClient = OpenApiClient(api_key)

#### 01_Local SSD 적재 방식

In [28]:
# fact_path = open_api_client.fetch_and_save_fact_period(
#     service_name=service_name,
#     start_date="2025-01-01",
#     end_date="2025-01-03",
#     base_path=base_fact_path
# )

#### 02_Cloud Object Storage(AWS S3) 적재 방식

In [29]:
aws_config = config["aws"]
BUCKET_NAME = aws_config["bucket_name"]
ACCESS_KEY = aws_config["access_key_id"]
SECRET_KEY = aws_config["secret_access_key"]
REGION = aws_config["region"]


os.environ["AWS_ACCESS_KEY_ID"] = ACCESS_KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = SECRET_KEY
os.environ["AWS_DEFAULT_REGION"] = REGION
os.environ["AWS_REGION"] = REGION

s3_base_uri = f"s3://{BUCKET_NAME}/seoul-public-bike/fact"

print(f"[S3 Mode] 데이터 레이크 팩트 영역 적재 가동 -> {s3_base_uri}")

# 2. 만능 팩트 엔진 가동 (base_uri 자리에 s3:// 주소 문자열을 주입
open_api_client.fetch_and_save_fact_period(
    service_name=service_name,
    start_date="2025-01-01",
    end_date="2025-01-03",
    base_path=s3_base_uri
)

print("-" * 80)
print(" [S3 Read Test] Polars로 S3 클라우드 파케이트 파일들 다이렉트 스캔 검증")
print("-" * 80)

# 3. S3 적재 결과 Polars 지연 연산(Lazy) 스캔 검증
s3_fact_glob = f"s3://{BUCKET_NAME}/seoul-public-bike/fact/{service_name}/**/*"

# 💡 첫 번째 칸에서 세팅했던 AWS 인증 토큰 정보(boto3용 변수들)를 그대로 바인딩
storage_options = {
    "aws_access_key_id": ACCESS_KEY,
    "aws_secret_access_key": SECRET_KEY,
    "aws_region": REGION,
}

# 로컬 폴더 대신 S3 경로와 인증 정보를 던져서 레이지(Lazy)하게 스캔 슛!
lazy_df_s3 = pl.scan_parquet(s3_fact_glob, storage_options=storage_options)
as_cleaned_df_s3 = lazy_df_s3.select(pl.all().name.to_lowercase())
final_df_s3 = as_cleaned_df_s3.collect(engine="streaming")

print(f"🔥 Cloud 스캔 성공! 최종 데이터 구조: {final_df_s3.shape}")
print(final_df_s3.head())

[S3 Mode] 데이터 레이크 팩트 영역 적재 가동 -> s3://my-fast-test-datalake-bucket-637423225965-ap-northeast-2-an/seoul-public-bike/fact

 [Fact Engine 가동] 기간: 2025-01-01 ~ 2025-01-03 (총 3일치 순회)
 ✅ 20250101 일자 하이브 파티션 최종 마감 및 메모리(Heap) 소멸 완료.
 ✅ 20250102 일자 하이브 파티션 최종 마감 및 메모리(Heap) 소멸 완료.
 ✅ 20250103 일자 하이브 파티션 최종 마감 및 메모리(Heap) 소멸 완료.

🎉 모든 기간의 트랜잭션(Fact) 데이터 [Extract ➡️ Load] 스트리밍 적재가 대성공으로 종료되었습니다.
--------------------------------------------------------------------------------
 [S3 Read Test] Polars로 S3 클라우드 파케이트 파일들 다이렉트 스캔 검증
--------------------------------------------------------------------------------
🔥 Cloud 스캔 성공! 최종 데이터 구조: (130277, 14)
shape: (5, 14)
┌────────────┬─────────┬──────────────┬───────────┬───┬───────────┬─────────────┬───────────┬──────┐
│ rent_dt    ┆ rent_id ┆ rent_nm      ┆ rent_type ┆ … ┆ move_time ┆ start_index ┆ end_index ┆ rnum │
│ ---        ┆ ---     ┆ ---          ┆ ---       ┆   ┆ ---       ┆ ---         ┆ ---       ┆ ---  │
│ str        ┆ str     ┆ str          ┆

In [30]:
# base_fact_path 경로에 여러 서비스네임, 여러 확장자가 있는 경우, 에러를 나기에, service_name 까지 지정 해줘야 한다
# scan_parquet = 이 폴더 깊숙이 숨어있는 파케이트 파일들을 다 찾아서 위아래로 이어 붙여라(Union)
lazy_df = pl.scan_parquet(base_fact_path / service_name / "**/*.parquet")     # ** = 하위폴더 전부 찾기
as_cleaned_df = lazy_df.select(pl.all().name.to_lowercase())
final_df = as_cleaned_df.collect(engine="streaming")
print(final_df)

shape: (130_277, 14)
┌────────────┬─────────┬─────────────┬───────────┬───┬───────────┬─────────────┬───────────┬───────┐
│ rent_dt    ┆ rent_id ┆ rent_nm     ┆ rent_type ┆ … ┆ move_time ┆ start_index ┆ end_index ┆ rnum  │
│ ---        ┆ ---     ┆ ---         ┆ ---       ┆   ┆ ---       ┆ ---         ┆ ---       ┆ ---   │
│ str        ┆ str     ┆ str         ┆ str       ┆   ┆ str       ┆ i64         ┆ i64       ┆ str   │
╞════════════╪═════════╪═════════════╪═══════════╪═══╪═══════════╪═════════════╪═══════════╪═══════╡
│ 2025-01-01 ┆ 01060   ┆ 1060.       ┆ 정기권    ┆ … ┆ 5         ┆ 0           ┆ 0         ┆ 1     │
│            ┆         ┆ 천일초교    ┆           ┆   ┆           ┆             ┆           ┆       │
│            ┆         ┆ 사거리      ┆           ┆   ┆           ┆             ┆           ┆       │
│ 2025-01-01 ┆ 00351   ┆ 351.        ┆ 정기권    ┆ … ┆ 56        ┆ 0           ┆ 0         ┆ 2     │
│            ┆         ┆ 청운초교 앞 ┆           ┆   ┆           ┆             ┆        

In [31]:
# =====================================================================
# ① 데이터의 날짜가 진짜 누락 없이 다 들어왔는지 날짜별 개수 확인
# =====================================================================
print("[진단 1] 날짜별 데이터 적재 건수 (누락/중복 체크)")
print(final_df["rent_dt"].value_counts())
print("\n" + "="*60 + "\n")

# =====================================================================
# ② 전체 컬럼들의 이름과 데이터 타입(str, i64 등) 명세서만 한눈에 보기
# =====================================================================
print("[진단 2] 폴라스 테이블 스키마 및 데이터 타입 명세서")
print(final_df.schema)
print("\n" + "="*60 + "\n")

# =====================================================================
# ③ 각 컬럼별로 NULL(비어있는 값)이 몇 개나 있는지 인프라 무결성 체크
# =====================================================================
print("🛡[진단 3] 컬럼별 결측치(NULL) 개수 검사")
print(final_df.null_count())
print("\n" + "="*60 + "\n")

# =====================================================================
# ④ 수치형 데이터들의 평균, 최소값, 최대값, 사분위수 통계치 요약해서 보기
# =====================================================================
print("[진단 4] 수치형 컬럼 기본 통계치 요약")
print(final_df.describe())

[진단 1] 날짜별 데이터 적재 건수 (누락/중복 체크)
shape: (3, 2)
┌────────────┬───────┐
│ rent_dt    ┆ count │
│ ---        ┆ ---   │
│ str        ┆ u32   │
╞════════════╪═══════╡
│ 2025-01-03 ┆ 47450 │
│ 2025-01-01 ┆ 34830 │
│ 2025-01-02 ┆ 47997 │
└────────────┴───────┘


[진단 2] 폴라스 테이블 스키마 및 데이터 타입 명세서
Schema({'rent_dt': String, 'rent_id': String, 'rent_nm': String, 'rent_type': String, 'gender_cd': String, 'age_type': String, 'use_cnt': String, 'exer_amt': String, 'carbon_amt': String, 'move_meter': String, 'move_time': String, 'start_index': Int64, 'end_index': Int64, 'rnum': String})


🛡[진단 3] 컬럼별 결측치(NULL) 개수 검사
shape: (1, 14)
┌─────────┬─────────┬─────────┬───────────┬───┬───────────┬─────────────┬───────────┬──────┐
│ rent_dt ┆ rent_id ┆ rent_nm ┆ rent_type ┆ … ┆ move_time ┆ start_index ┆ end_index ┆ rnum │
│ ---     ┆ ---     ┆ ---     ┆ ---       ┆   ┆ ---       ┆ ---         ┆ ---       ┆ ---  │
│ u32     ┆ u32     ┆ u32     ┆ u32       ┆   ┆ u32       ┆ u32         ┆ u32       ┆ u32  │
╞═════

In [32]:
from utils.db_client import BaseDBClient, PostgresDBClient
db_config = config.get("database", {})
db_clidnt:BaseDBClient = PostgresDBClient(**db_config)

df_url = db_clidnt.get_connection_uri()

In [33]:
engine = create_engine(df_url)

# 💡 테이블 이름에서 ods_를 걷어내고, 폴라스 타입과 1:1로 매핑했습니다.
temp_ods = """
CREATE SCHEMA IF NOT EXISTS temp;

DROP TABLE IF EXISTS temp.fact_bike_rent_daily;

CREATE TABLE temp.fact_bike_rent_daily (
    rent_dt TEXT,
    rent_id TEXT,
    rent_nm TEXT,
    rent_type TEXT,
    gender_cd TEXT,
    age_type TEXT,
    use_cnt TEXT,
    exer_amt TEXT,
    carbon_amt TEXT,
    move_meter TEXT,
    move_time TEXT,
    start_index BIGINT,
    end_index BIGINT,
    rnum TEXT
);
"""

with engine.connect() as conn:
    conn.execute(text(temp_ods))
    conn.commit()

# 💡 하위 적재 코드의 테이블명도 깔끔하게 매치시켜 줍니다.
table_with_schema = "temp.fact_bike_rent_daily"

final_df.write_database(
    table_name=table_with_schema,
    connection=df_url,
    if_table_exists="append",
    engine="adbc"  # ADBC 엔진이 자체적으로 하드웨어 레벨에서 데이터를 가장 안전하고 빠른 최적의 크기(chunk_size)로 조각조각 스트리밍하며 DB에 밀어 넣습니다
)

OperationalError: (psycopg2.OperationalError) connection to server at "192.168.56.10", port 5432 failed: timeout expired

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# =======================================================================
# [1단계 파이프라인] TEMP_ODS ➡️ ODS 적재용 변수
# =======================================================================
ods_source_table = "temp.fact_bike_rent_daily"
ods_target_table = "ods.fact_bike_rent_daily"

sql_file_path = ROOT_DIR / "src" / "ods" / "fact_bike_rent_daily.sql"
sql_template = sql_file_path.read_text(encoding="utf-8")
execute_ods_table = sql_template.format(
    target_table=ods_target_table,
    source_table=ods_source_table
)

with engine.connect() as conn:
    conn.execute(text(execute_ods_table))
    conn.commit()

print("ODS 적재 및 원천 삭제 데이터 추적 완벽 성공!")

In [ ]:
# =======================================================================
# [1단계 파이프라인] TEMP_ODS ➡️ ODS 적재용 변수
# =======================================================================
dw_source_table = "ods.fact_bike_rent_daily"
dw_target_table = "dw.fact_bike_rent_daily"

temp_ods = f"""
delete from {dw_target_table} where (
    rent_dt between '{start_date_formatted}' and '{end_date_formatted}'
);
"""


sql_file_path = ROOT_DIR / "src" / "dw" / "fact_bike_rent_daily.sql"
sql_template = sql_file_path.read_text(encoding="utf-8")
execute_dw_table = sql_template.format(
    target_table=dw_target_table,
    source_table=dw_source_table,
    start_date = start_date_formatted,
    end_date = end_date_formatted,
)

# =======================================================================
# [실행 레이어] 정의된 순서 그대로 트랜잭션 실행!
# =======================================================================
with engine.connect() as conn:
    with conn.begin(): # 안전 벨트 매고 하나의 세트로 묶어서 실행
        # 🧹 위에서 먼저 정의한 temp_ods(삭제)를 1등으로 실행!
        print(f"🧹 [1단계] {start_date_formatted} ~ {end_date_formatted} 기존 DW 데이터 삭제 중...")
        conn.execute(text(temp_ods))

        # 🚀 그다음 조립한 execute_dw_table(적재)를 2등으로 실행!
        print("🚀 [2단계] ODS 데이터를 세탁하여 DW로 초압축 적재 중...")
        conn.execute(text(execute_dw_table))

print("✅ dw 적재 및 원천 삭제 데이터 추적 완벽 성공!")